In [ ]:
# !PYTHONPATH=../src python ../main.py


Running Analysis for: 2008 Financial Crisis - Concentrated Financials
Time Horizon: 2007-01-01 to 2009-12-31
Confidence Level: 95.0%
  - Historical VaR (%):        -0.0554
  - Historical VaR ($):        $55,370.51
  - Parametric (Normal) VaR (%): 0.0647
  - Parametric (Normal) VaR ($): $64,682.79

Running Analysis for: 2008 Financial Crisis - Multi-Asset Diverse
Time Horizon: 2007-01-01 to 2009-12-31
Confidence Level: 95.0%
  - Historical VaR (%):        -0.0163
  - Historical VaR ($):        $16,342.88
  - Parametric (Normal) VaR (%): 0.0187
  - Parametric (Normal) VaR ($): $18,654.43


In [ ]:
# !PYTHONPATH=../src python ../src/stress_engine/monte_carlo.py

Executing standalone Monte Carlo stress tests...
Simulation results successfully serialized to disk.


In [ ]:
# !PYTHONPATH=../src python ../src/stress_engine/monte_carlo_shock.py

Executing Shock-Enabled Monte Carlo Stress Tests...
Shock simulation results successfully saved to synthetic_monte_carlo_shock_results.parquet


In [2]:
import sys
from pathlib import Path

# Setup path to include src/
src_path = Path("../src").resolve()
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from stress_engine.portfolio import PortfolioVaR


def run_portfolio_analysis() -> None:
    portfolios = [
        PortfolioVaR(
            name="2008 Financial Crisis - Concentrated Financials",
            tickers=["SPY", "C", "BAC", "XLF", "GS"],
            weights=[0.2, 0.2, 0.2, 0.2, 0.2],
            start_date="2007-01-01",
            end_date="2009-12-31",
            initial_capital=1_000_000,
            confidence_level=0.95,
        ),
        PortfolioVaR(
            name="2008 Financial Crisis - Multi-Asset Diverse",
            tickers=["SPY", "TLT", "GLD", "DBC"],
            weights=[0.4, 0.3, 0.2, 0.1],
            start_date="2007-01-01",
            end_date="2009-12-31",
            initial_capital=1_000_000,
            confidence_level=0.95,
        ),
    ]

    for port in portfolios:
        print("\n==================================================")
        print(f"Running Analysis for: {port.name}")
        print(f"Time Horizon: {port.start_date} to {port.end_date}")
        print("==================================================")

        metrics = port.run_analysis()

        print(f"Confidence Level: {port.confidence_level * 100}%")
        print(f"  - Historical VaR (%):        {metrics['hist_pct']:.4f}")
        print(
            f"  - Historical VaR ($):        ${abs(float(metrics['hist_dollar'])):,.2f}"
        )
        print(f"  - Parametric (Normal) VaR (%): {metrics['param_pct']:.4f}")
        print(
            f"  - Parametric (Normal) VaR ($): ${abs(float(metrics['param_dollar'])):,.2f}"
        )


run_portfolio_analysis()


Running Analysis for: 2008 Financial Crisis - Concentrated Financials
Time Horizon: 2007-01-01 to 2009-12-31
Confidence Level: 95.0%
  - Historical VaR (%):        -0.0554
  - Historical VaR ($):        $55,370.71
  - Parametric (Normal) VaR (%): 0.0647
  - Parametric (Normal) VaR ($): $64,682.79

Running Analysis for: 2008 Financial Crisis - Multi-Asset Diverse
Time Horizon: 2007-01-01 to 2009-12-31
Confidence Level: 95.0%
  - Historical VaR (%):        -0.0163
  - Historical VaR ($):        $16,342.72
  - Parametric (Normal) VaR (%): 0.0187
  - Parametric (Normal) VaR ($): $18,654.43


In [3]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Setup path to include src/
src_path = Path("../src").resolve()
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from stress_engine.monte_carlo import generate_parameter_grid, run_extended_monte_carlo


def run_standard_mc_pipeline() -> pd.DataFrame:
    grid = generate_parameter_grid()
    ledger: list[dict[str, float | str]] = []

    print("Executing standard standalone Monte Carlo stress tests...")
    for inst_name, profile in grid.items():
        prob, max_dds = run_extended_monte_carlo(profile, n_paths=2000, n_days=90)
        ledger.append(
            {
                "Institution": inst_name,
                "Annual_Vol": profile["annual_vol"],
                "Tail_DF": profile["t_df"],
                "GJR_Gamma": profile["gjr_gamma"],
                "Distress_Probability": prob,
                "Mean_Max_DD": float(np.mean(max_dds)),
                "P95_Max_DD": float(np.percentile(max_dds, 5)),
            }
        )

    df_out = pd.DataFrame(ledger)
    output_dir = Path("../data/raw")
    output_dir.mkdir(parents=True, exist_ok=True)
    output_file = output_dir / "synthetic_monte_carlo_results.parquet"

    df_out.to_parquet(output_file, index=False)
    print(f"Standard simulation results saved to: {output_file}")
    return df_out


df_standard_results = run_standard_mc_pipeline()
df_standard_results.head()

Executing standard standalone Monte Carlo stress tests...
Standard simulation results saved to: ../data/raw/synthetic_monte_carlo_results.parquet


,Institution,Annual_Vol,Tail_DF,GJR_Gamma,Distress_Probability,Mean_Max_DD,P95_Max_DD
0,Vol-Low_Tail-Fat_Asym-Low,0.1,3.5,0.01,0.0,-0.004034,-0.007906
1,Vol-Low_Tail-Fat_Asym-Norm,0.1,3.5,0.05,0.0,-0.003965,-0.007742
2,Vol-Low_Tail-Fat_Asym-High,0.1,3.5,0.15,0.0,-0.004033,-0.007627
3,Vol-Low_Tail-Norm_Asym-Low,0.1,8.0,0.01,0.0,-0.004295,-0.008078
4,Vol-Low_Tail-Norm_Asym-Norm,0.1,8.0,0.05,0.0,-0.004209,-0.007817


In [ ]:
# Mean_man_DD is normal

In [5]:
import sys
from pathlib import Path

import pandas as pd

# Setup path to include src/
src_path = Path("../src").resolve()
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from stress_engine.monte_carlo_shock import (
    generate_parameter_grid,
    run_shock_monte_carlo,
)


def run_shock_mc_pipeline() -> pd.DataFrame:
    grid = generate_parameter_grid()
    ledger: list[dict[str, float | str]] = []

    print("Executing Shock-Enabled Monte Carlo Stress Tests...")
    for inst_name, profile in grid.items():
        prob, max_dds = run_shock_monte_carlo(profile, n_paths=2000, n_days=90)
        ledger.append(
            {
                "Institution": inst_name,
                "Annual_Vol": profile["annual_vol"],
                "Tail_DF": profile["t_df"],
                "GJR_Gamma": profile["gjr_gamma"],
                "Distress_Probability": prob,
                "Mean_Max_DD": float(np.mean(max_dds)),
                "P95_Max_DD": float(np.percentile(max_dds, 5)),
            }
        )

    df_out = pd.DataFrame(ledger)
    output_dir = Path("../data/raw")
    output_dir.mkdir(parents=True, exist_ok=True)
    output_file = output_dir / "synthetic_monte_carlo_shock_results.parquet"

    df_out.to_parquet(output_file, index=False)
    print(f"Shock simulation results saved to: {output_file}")
    return df_out


df_shock_results = run_shock_mc_pipeline()
df_shock_results.sort_values(by="Distress_Probability", ascending=False).head(10)

Executing Shock-Enabled Monte Carlo Stress Tests...
Shock simulation results saved to: ../data/raw/synthetic_monte_carlo_shock_results.parquet


,Institution,Annual_Vol,Tail_DF,GJR_Gamma,Distress_Probability,Mean_Max_DD,P95_Max_DD
0,Vol-Low_Tail-Fat_Asym-Low,0.1,3.5,0.01,0.0,-0.148706,-0.152296
1,Vol-Low_Tail-Fat_Asym-Norm,0.1,3.5,0.05,0.0,-0.173891,-0.177274
2,Vol-Low_Tail-Fat_Asym-High,0.1,3.5,0.15,0.0,-0.233642,-0.236819
3,Vol-Low_Tail-Norm_Asym-Low,0.1,8.0,0.01,0.0,-0.148900,-0.152399
4,Vol-Low_Tail-Norm_Asym-Norm,0.1,8.0,0.05,0.0,-0.173972,-0.177206
5,Vol-Low_Tail-Norm_Asym-High,0.1,8.0,0.15,0.0,-0.233699,-0.236963
6,Vol-Low_Tail-Thin_Asym-Low,0.1,30.0,0.01,0.0,-0.148828,-0.152374
7,Vol-Low_Tail-Thin_Asym-Norm,0.1,30.0,0.05,0.0,-0.174063,-0.177636
8,Vol-Low_Tail-Thin_Asym-High,0.1,30.0,0.15,0.0,-0.233741,-0.236798
9,Vol-Norm_Tail-Fat_Asym-Low,0.2,3.5,0.01,0.0,-0.151696,-0.158948


In [ ]:
# Institution	Annual_Vol	Tail_DF	GJR_Gamma	Distress_Probability	Mean_Max_DD	P95_Max_DD
# 0	Vol-Low_Tail-Fat_Asym-Low	0.1	3.5	0.01	0.0	-0.148706	-0.152296
# 1	Vol-Low_Tail-Fat_Asym-Norm	0.1	3.5	0.05	0.0	-0.173891	-0.177274
# 2	Vol-Low_Tail-Fat_Asym-High	0.1	3.5	0.15	0.0	-0.233642	-0.236819
# 3	Vol-Low_Tail-Norm_Asym-Low	0.1	8.0	0.01	0.0	-0.148900	-0.152399
# 4	Vol-Low_Tail-Norm_Asym-Norm	0.1	8.0	0.05	0.0	-0.173972	-0.177206
# 5	Vol-Low_Tail-Norm_Asym-High	0.1	8.0	0.15	0.0	-0.233699	-0.236963
# 6	Vol-Low_Tail-Thin_Asym-Low	0.1	30.0	0.01	0.0	-0.148828	-0.152374
# 7	Vol-Low_Tail-Thin_Asym-Norm	0.1	30.0	0.05	0.0	-0.174063	-0.177636
# 8	Vol-Low_Tail-Thin_Asym-High	0.1	30.0	0.15	0.0	-0.233741	-0.236798
# 9	Vol-Norm_Tail-Fat_Asym-Low	0.2	3.5	0.01	0.0	-0.151696	-0.158948

# Mean_man_DD has spiked, 24% - 15% shock simulation compared to 1.5% - 0.4% non-shock simulation. This is a significant increase in the probability of distress, indicating that the shock scenario has a substantial impact on the risk profile of the institutions being analyzed.